### Recreating results from Zarifkar et al

In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import StandardScaler

In [ ]:
# Load data
df_featues = pd.read_csv("../Modified_Pupilometri/pupilometry_features_right_noCHerror.csv")

df_featues = df_featues[df_featues['redcap_repeat_instance'] == 1]
df_featues=df_featues.reset_index(drop=True)

feature_cols = [
    "record_id",
    "SECONDS",
    "FOUR",
    "pupil_constriction",
    "pupil_dilation"
]

In [ ]:
df_final = df_featues[feature_cols]

# Map SECONDS to numeric value
mapping = {
    "C": "0",
    "U": "1",
    "M-": "2",
    "M+": "3",
    "E": "4"
}

df_final["SECONDS_numeric"] =  pd.to_numeric(df_final["SECONDS"].map(mapping), errors="coerce")

df_final["FOUR"] = pd.to_numeric(df_final["FOUR"], errors="coerce")

df_final = df_final.dropna()

In [76]:
df_final["SECONDS"].value_counts()

SECONDS
C     197
U      35
M-     14
M+      3
E       1
Name: count, dtype: int64

In [ ]:
# Spearmans rank correlation with FOUR
features = [
    "record_id",
    "SECONDS_numeric",
    "FOUR",
    "pupil_constriction",
    "pupil_dilation"
]

for feature in features:
    corr, p = spearmanr(df_final[feature], df_final["FOUR"])
    print(f"{feature}: corr={corr:.3f}, p={p:.4f}")

record_id: corr=0.041, p=0.5175
SECONDS_numeric: corr=0.685, p=0.0000
FOUR: corr=1.000, p=0.0000
pupil_constriction: corr=0.551, p=0.0000
pupil_dilation: corr=0.444, p=0.0000


In [ ]:
# Spearmans rank correlation with SECONDS
for feature in features:
    corr, p = spearmanr(df_final[feature], df_final["SECONDS_numeric"])
    print(f"{feature}: corr={corr:.3f}, p={p:.4f}")

record_id: corr=0.017, p=0.7930
SECONDS_numeric: corr=1.000, p=0.0000
FOUR: corr=0.685, p=0.0000
pupil_constriction: corr=0.429, p=0.0000
pupil_dilation: corr=0.354, p=0.0000


In [ ]:
# Define model for PLR
features = [
    "pupil_constriction"
]

X_all = df_final[features]
y = df_final["SECONDS_numeric"]


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
X_scaled = pd.DataFrame(X_scaled, columns=X_all.columns, index=X_all.index)

model_all = OrderedModel(y, X_scaled, distr='probit') 
res_all = model_all.fit(method='bfgs')
print(res_all.summary())

Optimization terminated successfully.
         Current function value: 0.606382
         Iterations: 22
         Function evaluations: 23
         Gradient evaluations: 23
                             OrderedModel Results                             
Dep. Variable:        SECONDS_numeric   Log-Likelihood:                -151.60
Model:                   OrderedModel   AIC:                             313.2
Method:            Maximum Likelihood   BIC:                             330.8
Date:                Sat, 20 Jun 2026                                         
Time:                        12:19:10                                         
No. Observations:                 250                                         
Df Residuals:                     245                                         
Df Model:                           1                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------

In [ ]:
# Define model for LOR
ALL_features = [
    "pupil_dilation"
]

X_all = df_final[ALL_features]
y = df_final["SECONDS_numeric"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
X_scaled = pd.DataFrame(X_scaled, columns=X_all.columns, index=X_all.index)

model_all = OrderedModel(y, X_scaled, distr='probit') 
res_all = model_all.fit(method='bfgs')
print(res_all.summary())

Optimization terminated successfully.
         Current function value: 0.657486
         Iterations: 21
         Function evaluations: 22
         Gradient evaluations: 22
                             OrderedModel Results                             
Dep. Variable:        SECONDS_numeric   Log-Likelihood:                -164.37
Model:                   OrderedModel   AIC:                             338.7
Method:            Maximum Likelihood   BIC:                             356.4
Date:                Sat, 20 Jun 2026                                         
Time:                        12:19:10                                         
No. Observations:                 250                                         
Df Residuals:                     245                                         
Df Model:                           1                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------

In [ ]:
# Define model for PLR/LOR
ALL_features = [
    "pupil_dilation","pupil_constriction"
]

X_all = df_final[ALL_features]
y = df_final["SECONDS_numeric"]



scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)
X_scaled = pd.DataFrame(X_scaled, columns=X_all.columns, index=X_all.index)

model_all = OrderedModel(y, X_scaled, distr='probit')  # can also try 'probit'
res_all = model_all.fit(method='bfgs')
print(res_all.summary())

Optimization terminated successfully.
         Current function value: 0.600184
         Iterations: 26
         Function evaluations: 27
         Gradient evaluations: 27
                             OrderedModel Results                             
Dep. Variable:        SECONDS_numeric   Log-Likelihood:                -150.05
Model:                   OrderedModel   AIC:                             312.1
Method:            Maximum Likelihood   BIC:                             333.2
Date:                Sat, 20 Jun 2026                                         
Time:                        12:19:10                                         
No. Observations:                 250                                         
Df Residuals:                     244                                         
Df Model:                           2                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------

In [82]:
df_final[["pupil_constriction", "pupil_dilation"]].corr()

,pupil_constriction,pupil_dilation
pupil_constriction,1.000000,0.813189
pupil_dilation,0.813189,1.000000
